# OCR at scale via Mistral's Batch API

---

## Apply OCR to Convert Images into Text

Optical Character Recognition (OCR) allows you to retrieve text data from images. With Mistral OCR, you can do this extremely fast and effectively, extracting text from hundreds and thousands of images (or PDFs).

In this simple cookbook, we will extract text from a set of images using two methods:
- [Without Batch Inference](#scrollTo=qmXyB3rPlXQW): Looping through the dataset, extracting text from each image, and saving the result.
- [With Batch Inference](#scrollTo=jYfWYjzTmixB): Leveraging Batch Inference to extract text with a 50% cost reduction.

---

### Used

- OCR
- Batch Inference


### Setup
First, let's install `mistralai` and `datasets`

In [1]:
!pip install mistralai==1.9.11

We can now set up our client. You can create an API key on our [Plateforme](https://console.mistral.ai/api-keys/).

In [2]:
from mistralai import Mistral

api_key = "GYoq28263LSn7SIO5WI7MYSNT1bqGbWJ"
client = Mistral(api_key=api_key)
ocr_model = "mistral-ocr-latest"

## Without Batch

As an example, let's use Mistral OCR to extract text from multiple images.

We will use a dataset containing raw image data. To send this data via an image URL, we need to encode it in base64. For more information, please visit our [Vision Documentation](https://docs.mistral.ai/capabilities/vision/#passing-a-base64-encoded-image).

In [ ]:
import base64
from io import BytesIO
from PIL import Image

def encode_image_data(image_data):
    try:
        # Ensure image_data is bytes
        if isinstance(image_data, bytes):
            # Directly encode bytes to base64
            return base64.b64encode(image_data).decode('utf-8')
        else:
            # Convert image data to bytes if it's not already
            buffered = BytesIO()
            image_data.save(buffered, format="JPEG")
            return base64.b64encode(buffered.getvalue()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None

For this demo, we will use a simple dataset containing numerous documents and scans in image format. Specifically, we will use the `HuggingFaceM4/DocumentVQA` dataset, loaded via the `datasets` library.

We will download only 100 samples for this demonstration.

In [ ]:
from datasets import load_dataset

n_samples = 100
dataset = load_dataset("HuggingFaceM4/DocumentVQA", split="train", streaming=True)
subset = list(dataset.take(n_samples))

With our subset of 100 samples ready, we can loop through each image to extract the text.

We will save the results in a new dataset and export it as a JSONL file.

In [ ]:
from tqdm import tqdm

ocr_dataset = []
for sample in tqdm(subset):
    image_data = sample['image']  # 'image' contains the actual image data

    # Encode the image data to base64
    base64_image = encode_image_data(image_data)
    image_url = f"data:image/jpeg;base64,{base64_image}"

    # Process the image using Mistral OCR
    response = client.ocr.process(
        model=ocr_model,
        document={
            "type": "image_url",
            "image_url": image_url,
        }
    )

    # Store the image data and OCR content in the new dataset
    ocr_dataset.append({
        'image': base64_image,
        'ocr_content': response.pages[0].markdown # Since we are dealing with single images, there will be only one page
    })

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [02:13<00:00,  1.33s/it]


In [ ]:
import json

with open('ocr_dataset.json', 'w') as f:
    json.dump(ocr_dataset, f, indent=4)

Perfect, we have extracted all text from the 100 samples. However, this process can be made more cost-efficient using Batch Inference.

## With Batch

To use Batch Inference, we need to create a JSONL file containing all the image data and request information for our batch.

Let's create a function called `create_batch_file` to handle this task by generating a file in the proper format.

In [ ]:
def create_batch_file(image_urls, output_file):
    with open(output_file, 'w') as file:
        for index, url in enumerate(image_urls):
            entry = {
                "custom_id": str(index),
                "body": {
                    "document": {
                        "type": "image_url",
                        "image_url": url
                    },
                    "include_image_base64": True
                }
            }
            file.write(json.dumps(entry) + '\n')

The next step involves encoding the data of each image into base64 and saving the URL of each image that will be used.

In [ ]:
image_urls = []
for sample in tqdm(subset):
    image_data = sample['image']  # 'image' contains the actual image data

    # Encode the image data to base64 and add the url to the list
    base64_image = encode_image_data(image_data)
    image_url = f"data:image/jpeg;base64,{base64_image}"
    image_urls.append(image_url)

 48%|████▊     | 48/100 [00:00<00:01, 41.07it/s]

100%|██████████| 100/100 [00:04<00:00, 24.48it/s]


We can now create our batch file.

In [ ]:
batch_file = "batch_file.jsonl"
create_batch_file(image_urls, batch_file)

With everything ready, we can upload it to the API.

In [ ]:
batch_data = client.files.upload(
    file={
        "file_name": batch_file,
        "content": open(batch_file, "rb")},
    purpose = "batch"
)

The file is uploaded, but the batch inference has not started yet. To initiate it, we need to create a job.

In [ ]:
created_job = client.batch.jobs.create(
    input_files=[batch_data.id],
    model=ocr_model,
    endpoint="/v1/ocr",
    metadata={"job_type": "testing"}
)

Our batch is ready and running!

We can retrieve information using the following method:

In [ ]:
retrieved_job = client.batch.jobs.get(job_id=created_job.id)
print(f"Status: {retrieved_job.status}")
print(f"Total requests: {retrieved_job.total_requests}")
print(f"Failed requests: {retrieved_job.failed_requests}")
print(f"Successful requests: {retrieved_job.succeeded_requests}")
print(
    f"Percent done: {round((retrieved_job.succeeded_requests + retrieved_job.failed_requests) / retrieved_job.total_requests, 4) * 100}%"
)

Status: QUEUED
Total requests: 100
Failed requests: 0
Successful requests: 0
Percent done: 0.0%


Let's automate this feedback loop and download the results once they are ready!

In [ ]:
import time
from IPython.display import clear_output

while retrieved_job.status in ["QUEUED", "RUNNING"]:
    retrieved_job = client.batch.jobs.get(job_id=created_job.id)

    clear_output(wait=True)  # Clear the previous output ( User Friendly )
    print(f"Status: {retrieved_job.status}")
    print(f"Total requests: {retrieved_job.total_requests}")
    print(f"Failed requests: {retrieved_job.failed_requests}")
    print(f"Successful requests: {retrieved_job.succeeded_requests}")
    print(
        f"Percent done: {round((retrieved_job.succeeded_requests + retrieved_job.failed_requests) / retrieved_job.total_requests, 4) * 100}%"
    )
    time.sleep(2)

Status: SUCCESS
Total requests: 100
Failed requests: 0
Successful requests: 100
Percent done: 100.0%


In [ ]:
client.files.download(file_id=retrieved_job.output_file)

<Response [200 OK]>

Done! With this method, you can perform OCR tasks in bulk in a very cost-effective way.

In [5]:
# Install PyMuPDF for PDF processing
!pip install pymupdf

import requests
import fitz # PyMuPDF
import base64
import json
from io import BytesIO
from PIL import Image # For the existing encode_image_data function to work with PIL images if needed
import time
from IPython.display import clear_output
from tqdm import tqdm # Assuming tqdm is available
import random

# --- Function Definitions (copied from earlier cells to ensure availability) ---
def encode_image_data(image_data):
    try:
        # Ensure image_data is bytes
        if isinstance(image_data, bytes):
            # Directly encode bytes to base64
            return base64.b64encode(image_data).decode('utf-8')
        else:
            # Convert image data to bytes if it's not already
            buffered = BytesIO()
            image_data.save(buffered, format="JPEG")
            return base64.b64encode(buffered.getvalue()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None

def create_batch_file(image_urls, output_file):
    with open(output_file, 'w') as file:
        for index, url in enumerate(image_urls):
            entry = {
                "custom_id": str(index),
                "body": {
                    "document": {
                        "type": "image_url",
                        "image_url": url
                    },
                    "include_image_base64": True
                }
            }
            file.write(json.dumps(entry) + '\n')
# --- End Function Definitions ---

# Define the GitHub URL for your PDF
# >>> IMPORTANT: Replace with your actual GitHub raw PDF URL <<<
pdf_github_url = "https://github.com/mozilla/pdf.js/raw/master/web/compressed.tracemonkey-pldi-09.pdf"
print(f"Starting OCR for PDF from URL: {pdf_github_url}")

# --- Step 1: Download the PDF ---
try:
    response = requests.get(pdf_github_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    pdf_document = fitz.open(stream=response.content, filetype="pdf")
    print(f"Downloaded PDF with {len(pdf_document)} pages.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading PDF: {e}")
    # Handle error, perhaps exit or raise a more specific exception
    exit()
except Exception as e:
    print(f"Error opening PDF: {e}")
    exit()

# --- Step 2: Convert PDF pages to base64 images and prepare image URLs (TEST MODE) ---
# Select 3 random pages for testing
num_pages_to_test = 3
all_page_numbers = list(range(len(pdf_document)))
random_page_indices = random.sample(all_page_numbers, min(num_pages_to_test, len(pdf_document)))

image_urls = []
print(f"Extracting images for OCR test on {len(random_page_indices)} random pages...")
for page_num in tqdm(random_page_indices, desc="Converting selected PDF pages to images"):
    page = pdf_document.load_page(page_num)
    pix = page.get_pixmap()
    # Convert pixmap to PNG bytes. You can use "jpeg" if preferred, but "png" is lossless.
    img_bytes = pix.tobytes("png")

    # Reuse the existing encode_image_data function (defined earlier in the notebook)
    base64_image = encode_image_data(img_bytes)
    if base64_image:
        image_urls.append(f"data:image/png;base64,{base64_image}")
    else:
        print(f"Warning: Could not encode image for page {page_num + 1}")
pdf_document.close()
print(f"Prepared {len(image_urls)} image URLs for OCR test.")

# --- Step 3: Create batch JSONL file ---
pdf_batch_file = "pdf_batch_file_test.jsonl"
# Reuse the existing create_batch_file function (defined earlier in the notebook)
create_batch_file(image_urls, pdf_batch_file)
print(f"Batch file '{pdf_batch_file}' created for {len(image_urls)} images.")

# --- Step 4: Upload batch file to Mistral ---
print("Uploading batch file to Mistral...")
pdf_batch_data = client.files.upload(
    file={
        "file_name": pdf_batch_file,
        "content": open(pdf_batch_file, "rb")},
    purpose = "batch"
)
print(f"Batch file uploaded with ID: {pdf_batch_data.id}")

# --- Step 5: Create batch job ---
print("Creating batch OCR job...")
pdf_created_job = client.batch.jobs.create(
    input_files=[pdf_batch_data.id],
    model=ocr_model, # ocr_model is already defined in the notebook
    endpoint="/v1/ocr",
    metadata={"job_type": "pdf_ocr_from_github_test"}
)
print(f"Batch job created with ID: {pdf_created_job.id}")

# --- Step 6: Monitor job status ---
print("Monitoring batch job status (this may take a while)...")
while pdf_created_job.status in ["QUEUED", "RUNNING"]:
    pdf_created_job = client.batch.jobs.get(job_id=pdf_created_job.id)
    clear_output(wait=True)
    print(f"Status: {pdf_created_job.status}")
    print(f"Total requests: {pdf_created_job.total_requests}")
    print(f"Failed requests: {pdf_created_job.failed_requests}")
    print(f"Successful requests: {pdf_created_job.succeeded_requests}")
    percent_done = 0
    if pdf_created_job.total_requests > 0:
        percent_done = round((pdf_created_job.succeeded_requests + pdf_created_job.failed_requests) / pdf_created_job.total_requests, 4) * 100
    print(f"Percent done: {percent_done}%")
    time.sleep(5) # Wait a bit longer to reduce API calls

if pdf_created_job.status == "SUCCESS":
    print("\nBatch OCR job completed successfully!")
else:
    print(f"\nBatch OCR job failed with status: {pdf_created_job.status}")
    # Optionally, retrieve errors if the job failed
    if pdf_created_job.error_file:
        error_response = client.files.download(file_id=pdf_created_job.error_file)
        print("Errors:")
        print(error_response.text)
    exit()

# --- Step 7: Download and save results ---
print("Downloading OCR results...")
pdf_ocr_results_response = client.files.download(file_id=pdf_created_job.output_file)
pdf_ocr_results_content = pdf_ocr_results_response.text # .read() n'est souvent pas nécessaire selon la version du SDK, .text suffit

pdf_ocr_dataset = []
for line in pdf_ocr_results_content.strip().split('\n'):
    if line:
        result = json.loads(line)

        # 1. On accède d'abord à 'response'
        response_data = result.get('response', {})
        # 2. Puis au 'body' de la réponse
        body_data = response_data.get('body', {})
        # 3. Enfin on extrait le markdown des pages
        pages = body_data.get('pages', [])

        ocr_text = pages[0].get('markdown', '') if pages else ''

        pdf_ocr_dataset.append({
            'page_number': result.get('custom_id', 'N/A'),
            'ocr_content': ocr_text
        })

output_json_filename = "pdf_ocr_output_test.json"
with open(output_json_filename, 'w', encoding='utf-8') as f:
    json.dump(pdf_ocr_dataset, f, indent=4, ensure_ascii=False)
print(f"OCR results for PDF saved to '{output_json_filename}'")

# Optionally, display a sample of the results
if pdf_ocr_dataset:
    print("\nSample OCR content for the first page:")
    print(pdf_ocr_dataset[0]['ocr_content'][:500]) # Print first 500 characters

Status: SUCCESS
Total requests: 3
Failed requests: 0
Successful requests: 3
Percent done: 100.0%

Batch OCR job completed successfully!
OCR results for PDF saved to 'pdf_ocr_output_test.json'

Sample OCR content for the first page:



In [12]:
pdf_ocr_dataset

[{'page_number': '0', 'ocr_content': ''},
 {'page_number': '1', 'ocr_content': ''},
 {'page_number': '2', 'ocr_content': ''}]